# MulSen-AD Raw Dataset Exploration

This notebook is the cleaned raw-data sanity check for the current MulSenDiff-X workflow.
It focuses on three useful questions:

1. Is the raw dataset layout complete and consistent across RGB, infrared, and point cloud modalities?
2. What do the train/test and annotation distributions look like after indexing?
3. What do aligned raw RGB, IR, and point-cloud samples look like for a small random subset of the dataset?

The gallery section uses the canonical `dataset_index.csv` only as an alignment registry. The rendered assets remain the raw RGB, IR, and STL point-cloud files.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import io
import random
import warnings

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import art3d

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    pd = None
    PANDAS_AVAILABLE = False

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    Image = None
    PIL_AVAILABLE = False

try:
    import trimesh
    TRIMESH_AVAILABLE = True
except ImportError:
    trimesh = None
    TRIMESH_AVAILABLE = False

plt.ioff()
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive, and thus cannot be shown')
NOTEBOOK_OUTPUT_DIR = Path('/tmp/mulsendiffx_notebook_outputs')
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_VERSION = '2026-04-07-raw-gallery-v2'

try:
    from IPython import get_ipython
    from IPython.display import Image as IPythonImage, display
except ImportError:
    get_ipython = None
    IPythonImage = None
    display = None


def configure_notebook_backend():
    ip = get_ipython() if get_ipython is not None else None
    if ip is None:
        return False
    try:
        ip.run_line_magic('matplotlib', 'inline')
    except Exception:
        pass
    return True


IN_NOTEBOOK = configure_notebook_backend()


def render_figure(fig, output_name='figure.png'):
    fig.tight_layout()
    output_path = NOTEBOOK_OUTPUT_DIR / output_name if output_name else None
    png_bytes = None
    if IN_NOTEBOOK and display is not None and IPythonImage is not None:
        buffer = io.BytesIO()
        fig.savefig(buffer, format='png', bbox_inches='tight', dpi=160)
        buffer.seek(0)
        png_bytes = buffer.getvalue()
        if output_path is not None:
            output_path.write_bytes(png_bytes)
        display(IPythonImage(data=png_bytes))
        buffer.close()
    else:
        if output_path is not None:
            fig.savefig(output_path, format='png', bbox_inches='tight', dpi=160)
        fig.canvas.draw()
    plt.close(fig)
    return output_path

print('Notebook version   :', NOTEBOOK_VERSION)
print('Pandas available   :', PANDAS_AVAILABLE)
print('Pillow available   :', PIL_AVAILABLE)
print('Trimesh available  :', TRIMESH_AVAILABLE)
print('Matplotlib backend :', plt.get_backend())
print('Output directory   :', NOTEBOOK_OUTPUT_DIR)

if not PIL_AVAILABLE:
    raise RuntimeError('Pillow is required for the raw sample visualization notebook.')
if not TRIMESH_AVAILABLE:
    raise RuntimeError('trimesh is required for STL mesh previews in this notebook.')


In [9]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'MulSen_AD').exists() and (candidate / 'src' / 'notebooks').exists():
            return candidate
    raise RuntimeError(f'Could not locate project root from {start}')


PROJECT_ROOT = find_project_root()
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'MulSen_AD'
INDEX_CSV = PROJECT_ROOT / 'data' / 'processed' / 'manifests' / 'dataset_index.csv'

assert RAW_ROOT.exists(), f'Missing raw dataset root: {RAW_ROOT}'
assert INDEX_CSV.exists(), f'Missing dataset index: {INDEX_CSV}'

rows = list(csv.DictReader(INDEX_CSV.open(newline='', encoding='utf-8')))
categories = sorted([path.name for path in RAW_ROOT.iterdir() if path.is_dir()])
modalities = ['RGB', 'Infrared', 'Pointcloud']
splits = ['train', 'test', 'GT']

print('Project root :', PROJECT_ROOT)
print('Raw root     :', RAW_ROOT)
print('Index CSV    :', INDEX_CSV)
print('Categories   :', len(categories))
print('Indexed rows :', len(rows))


Project root : /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026
Raw root     : /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026/data/raw/MulSen_AD
Index CSV    : /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026/data/processed/manifests/dataset_index.csv
Categories   : 15
Indexed rows : 2035


## Inventory and Coverage

The first pass verifies the category inventory, raw split structure, and annotation coverage before any modeling assumptions are made.


In [10]:
def count_files(path):
    return sum(1 for file_path in path.rglob('*') if file_path.is_file()) if path.exists() else 0


def list_dirs(path):
    return sorted([subdir.name for subdir in path.iterdir() if subdir.is_dir()]) if path.exists() else []


def gather_category_summary(category_name):
    category_path = RAW_ROOT / category_name
    summary = {
        'category': category_name,
        'rgb_train': count_files(category_path / 'RGB' / 'train'),
        'rgb_test': count_files(category_path / 'RGB' / 'test'),
        'ir_train': count_files(category_path / 'Infrared' / 'train'),
        'ir_test': count_files(category_path / 'Infrared' / 'test'),
        'pc_train': count_files(category_path / 'Pointcloud' / 'train'),
        'pc_test': count_files(category_path / 'Pointcloud' / 'test'),
        'rgb_gt': count_files(category_path / 'RGB' / 'GT'),
        'ir_gt': count_files(category_path / 'Infrared' / 'GT'),
        'pc_gt': count_files(category_path / 'Pointcloud' / 'GT'),
        'rgb_test_labels': list_dirs(category_path / 'RGB' / 'test'),
    }
    return summary


category_summary_rows = [gather_category_summary(category_name) for category_name in categories]

if PANDAS_AVAILABLE:
    summary_df = pd.DataFrame(category_summary_rows)
    display(summary_df)
else:
    for row in category_summary_rows:
        print(row)


,category,rgb_train,rgb_test,ir_train,ir_test,pc_train,pc_test,rgb_gt,ir_gt,pc_gt,rgb_test_labels
0,button_cell,90,41,90,41,90,41,31,25,15,"[color, foreign_body, good, hole, scratch]"
1,capsule,64,58,64,58,64,58,48,46,33,"[broken_inside, broken_outside, color, crack, ..."
2,cotton,78,50,78,50,78,50,14,39,40,"[broken, crack, crease, good, hole, scratch]"
3,cube,110,51,110,51,110,51,39,22,33,"[broken, color, foreign_body, good, hole, scra..."
4,flat_pad,90,40,90,40,90,40,35,24,23,"[broken, color, foreign_body, good, scratch]"
5,light,110,46,110,46,110,46,19,27,21,"[broken, color, detachment_inside, foreign_bod..."
6,nut,118,39,118,39,118,39,35,13,14,"[broken, color, foreign_body, good, scratch]"
7,piggy,110,40,110,40,110,40,35,28,18,"[broken, color, crack, good, hole, scratch]"
8,plastic_cylinder,90,38,90,38,90,38,23,25,21,"[broken, color, foreign_body, good, hole, scra..."
9,screen,69,42,69,42,69,42,34,23,32,"[broken, crack, good, label, scratch]"


In [11]:
split_counts = Counter(row['split'] for row in rows)
anomaly_counts = Counter(row['is_anomalous'] for row in rows)
extension_counts = Counter()
gt_coverage = defaultdict(lambda: Counter())

for category_name in categories:
    for modality in modalities:
        modality_path = RAW_ROOT / category_name / modality
        for file_path in modality_path.rglob('*'):
            if file_path.is_file():
                extension_counts[file_path.suffix.lower()] += 1

for row in rows:
    category = row['category']
    gt_coverage[category]['records'] += 1
    gt_coverage[category]['rgb_gt_available'] += int(row['rgb_gt_available'] == 'True')
    gt_coverage[category]['ir_gt_available'] += int(row['ir_gt_available'] == 'True')
    gt_coverage[category]['pointcloud_gt_available'] += int(row['pointcloud_gt_available'] == 'True')

print('Split counts     :', dict(split_counts))
print('Anomaly counts   :', dict(anomaly_counts))
print('File extensions  :', dict(extension_counts))

gt_rows = []
for category_name in categories:
    entry = {'category': category_name, **gt_coverage[category_name]}
    gt_rows.append(entry)

if PANDAS_AVAILABLE:
    gt_df = pd.DataFrame(gt_rows).fillna(0).sort_values('category').reset_index(drop=True)
    display(gt_df)
else:
    for row in gt_rows:
        print(row)


Split counts     : {'train': 1391, 'test': 644}
Anomaly counts   : {'False': 1541, 'True': 494}
File extensions  : {'.png': 4819, '.csv': 87, '.stl': 2035, '.txt': 342}


,category,records,rgb_gt_available,ir_gt_available,pointcloud_gt_available
0,button_cell,131,26,25,15
1,capsule,122,41,46,33
2,cotton,128,8,39,40
3,cube,161,33,22,33
4,flat_pad,130,30,24,23
5,light,156,12,27,21
6,nut,157,29,13,14
7,piggy,150,29,28,18
8,plastic_cylinder,128,17,25,21
9,screen,111,29,23,32


## Raw Asset Format Checks

A robust loader needs to know exactly what it is loading. This section inspects representative RGB, IR, STL, and point-cloud GT assets.


In [12]:
sample_paths = {
    'rgb_train': RAW_ROOT / 'button_cell' / 'RGB' / 'train' / '0.png',
    'ir_train': RAW_ROOT / 'button_cell' / 'Infrared' / 'train' / '0.png',
    'rgb_gt': RAW_ROOT / 'capsule' / 'RGB' / 'GT' / 'crack' / '0.png',
    'pointcloud_train': RAW_ROOT / 'button_cell' / 'Pointcloud' / 'train' / '0.stl',
    'pointcloud_gt': RAW_ROOT / 'button_cell' / 'Pointcloud' / 'GT' / 'foreign_body' / '0.txt',
}

for name, path in sample_paths.items():
    print(f'\n{name}: {path}')
    print('exists =', path.exists())
    if not path.exists():
        continue
    if path.suffix.lower() == '.png':
        image = Image.open(path)
        print('mode =', image.mode, '| size =', image.size)
    elif path.suffix.lower() == '.stl':
        mesh = trimesh.load(path, force='mesh')
        if not isinstance(mesh, trimesh.Trimesh) or len(mesh.vertices) == 0 or len(mesh.faces) == 0:
            raise RuntimeError(f'Trimesh could not read triangle mesh from {path}')
        vertices = np.asarray(mesh.vertices)
        triangles = np.asarray(mesh.faces)
        bounds = [vertices.min(axis=0).tolist(), vertices.max(axis=0).tolist()]
        print('vertices =', vertices.shape[0], '| faces =', triangles.shape[0])
        print('bounds =', bounds)
    elif path.suffix.lower() == '.txt':
        lines = [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
        print('annotated rows =', len(lines))
        print('preview =', lines[:3])

sampled_dimensions = defaultdict(set)
for category_name in categories[:5]:
    rgb_path = next((RAW_ROOT / category_name / 'RGB' / 'train').glob('*.png'))
    ir_path = next((RAW_ROOT / category_name / 'Infrared' / 'train').glob('*.png'))
    sampled_dimensions['RGB'].add(Image.open(rgb_path).size)
    sampled_dimensions['Infrared'].add(Image.open(ir_path).size)

print('\nUnique sampled dimensions:')
for modality, dims in sampled_dimensions.items():
    print(modality, sorted(dims))



rgb_train: /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026/data/raw/MulSen_AD/button_cell/RGB/train/0.png
exists = True
mode = RGBA | size = (1280, 960)

ir_train: /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026/data/raw/MulSen_AD/button_cell/Infrared/train/0.png
exists = True
mode = RGB | size = (640, 480)

rgb_gt: /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026/data/raw/MulSen_AD/capsule/RGB/GT/crack/0.png
exists = True
mode = L | size = (1280, 960)

pointcloud_train: /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026/data/raw/MulSen_AD/button_cell/Pointcloud/train/0.stl
exists = True
vertices = 93829 | faces = 187658
bounds = [[-117.37459564208984, -324.9732666015625, 2075.111083984375], [-106.38883972167969, -314.73974609375, 2086.54443359375]]

pointcloud_gt: /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSen

## Random Raw Multimodal Gallery

The gallery below draws **three reproducible random dataset items** from the canonical indexed registry and renders the aligned raw modalities as one row per item.


In [13]:
SEED = 7
NUM_SAMPLES = 3
CATEGORY_FILTER = None
SPLIT_FILTER = None
POINTCLOUD_MAX_POINTS = 5000
POINTCLOUD_ELEV = 25
POINTCLOUD_AZIM = 45

print({
    'SEED': SEED,
    'NUM_SAMPLES': NUM_SAMPLES,
    'CATEGORY_FILTER': CATEGORY_FILTER,
    'SPLIT_FILTER': SPLIT_FILTER,
    'POINTCLOUD_MAX_POINTS': POINTCLOUD_MAX_POINTS,
})


{'SEED': 7, 'NUM_SAMPLES': 3, 'CATEGORY_FILTER': None, 'SPLIT_FILTER': None, 'POINTCLOUD_MAX_POINTS': 5000}


In [ ]:
def filtered_rows(all_rows, category_filter=None, split_filter=None):
    selected = []
    for row in all_rows:
        if category_filter and row['category'] != category_filter:
            continue
        if split_filter and row['split'] != split_filter:
            continue
        if not row['rgb_path'] or not row['ir_path'] or not row['pointcloud_path']:
            continue
        selected.append(row)
    return selected


def load_stl_mesh_arrays(path, max_triangles=8000, seed=0):
    mesh = trimesh.load(path, force='mesh')
    if not isinstance(mesh, trimesh.Trimesh):
        raise RuntimeError(f'Expected a triangle mesh in {path}, but got {type(mesh).__name__}')
    if mesh.vertices is None or mesh.faces is None or len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        raise RuntimeError(f'Empty vertices or triangles found in {path}')

    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.faces)

    if len(triangles) > max_triangles:
        rng = np.random.default_rng(seed)
        indices = rng.choice(len(triangles), size=max_triangles, replace=False)
        triangles = triangles[indices]

    return vertices, vertices[triangles]


gallery_pool = filtered_rows(rows, category_filter=CATEGORY_FILTER, split_filter=SPLIT_FILTER)
if len(gallery_pool) < NUM_SAMPLES:
    raise RuntimeError(f'Requested {NUM_SAMPLES} samples but only found {len(gallery_pool)} rows after filtering.')

rng = random.Random(SEED)
gallery_rows = rng.sample(gallery_pool, NUM_SAMPLES)

metadata_rows = []
fig = plt.figure(figsize=(15, 4.5 * NUM_SAMPLES))

for row_index, row in enumerate(gallery_rows):
    rgb = np.asarray(Image.open(PROJECT_ROOT / row['rgb_path']).convert('RGB'))
    ir = np.asarray(Image.open(PROJECT_ROOT / row['ir_path']).convert('L'))
    vertices, triangle_vertices = load_stl_mesh_arrays(
        PROJECT_ROOT / row['pointcloud_path'],
        max_triangles=POINTCLOUD_MAX_POINTS,
        seed=SEED + row_index,
    )

    metadata_rows.append({
        'category': row['category'],
        'split': row['split'],
        'defect_label': row['defect_label'],
        'sample_id': row['sample_id'],
        'is_anomalous': row['is_anomalous'],
    })

    ax_rgb = fig.add_subplot(NUM_SAMPLES, 3, row_index * 3 + 1)
    ax_rgb.imshow(rgb)
    ax_rgb.set_title('RGB')
    ax_rgb.set_xticks([])
    ax_rgb.set_yticks([])
    ax_rgb.set_ylabel(
        '\n'.join([
            row['category'],
            f"{row['split']} / {row['defect_label']}",
            f"id={row['sample_id']}",
            f"anomalous={row['is_anomalous']}",
        ]),
        rotation=0,
        ha='right',
        va='center',
        labelpad=56,
    )

    ax_ir = fig.add_subplot(NUM_SAMPLES, 3, row_index * 3 + 2)
    ax_ir.imshow(ir, cmap='inferno')
    ax_ir.set_title('Infrared')
    ax_ir.set_xticks([])
    ax_ir.set_yticks([])

    ax_pc = fig.add_subplot(NUM_SAMPLES, 3, row_index * 3 + 3, projection='3d')
    mesh_collection = art3d.Poly3DCollection(
        triangle_vertices,
        facecolor='#7ba6d9',
        edgecolor='#355c8c',
        linewidth=0.05,
        alpha=0.85,
    )
    ax_pc.add_collection3d(mesh_collection)
    ax_pc.view_init(elev=POINTCLOUD_ELEV, azim=POINTCLOUD_AZIM)
    mins = vertices.min(axis=0)
    maxs = vertices.max(axis=0)
    extent = maxs - mins
    extent[extent == 0] = 1.0
    ax_pc.set_xlim(mins[0], maxs[0])
    ax_pc.set_ylim(mins[1], maxs[1])
    ax_pc.set_zlim(mins[2], maxs[2])
    try:
        ax_pc.set_box_aspect(extent)
    except Exception:
        pass
    ax_pc.set_title('Point cloud')
    ax_pc.set_axis_off()

gallery_output_path = render_figure(fig, output_name='01_raw_multimodal_gallery.png')
print('Saved gallery PNG :', gallery_output_path)

if PANDAS_AVAILABLE:
    display(pd.DataFrame(metadata_rows))
else:
    for row in metadata_rows:
        print(row)


,category,split,defect_label,sample_id,is_anomalous
0,flat_pad,test,scratch,1,True
1,zipper,train,good,38,False
2,cotton,train,good,59,False


## Engineering Takeaways

- RGB, infrared, and point-cloud assets are present as separate raw modalities and should continue to be treated as registered views of the same part instance.
- The processed `dataset_index.csv` is the safest registry for aligned sampling because it already resolves the three raw paths for each indexed item.
- STL point clouds are large enough to need subsampling for notebook previews, so the visualization deliberately favors fast static inspection over interactive rendering.
- This notebook should remain focused on raw-data QA and paper-figure scouting, not on downstream descriptor or detector analysis.
